# Map Buzzer/LED Signals To Session Time

This notebook demonstrates the intended analysis pattern for synchronized multimodal data: process each channel in its native source coordinates first, then use the synchronization tables to map source samples, frames, and detected events onto the shared session timebase.

## Coordinate Systems

- **File-local** coordinates describe data inside one acquired file. For audio, this is time in seconds from the start of one WAV file. For video, this is frame index from the start of one video file.
- **Source** coordinates describe one continuous acquisition channel after its files are ordered and concatenated. For audio, this is the channel-wide sample index. For video, this is the channel-wide frame index.
- **Session** coordinates describe the shared synchronization clock. Session time is defined by the sync pulse train, with the first valid pulse at `t = 0`.

The workflow below keeps these layers separate: detect events per file, convert file-local events into source indices, then map source indices to session time.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
import soundfile as sf

from multimodal_sync.config import config_root, get_sync_rate_hz, load_config
from multimodal_sync.continuous import map_source_signal_to_session_time
from multimodal_sync.events import (
    detect_threshold_intervals,
    file_local_index_events_to_source_indices,
    file_local_time_events_to_source_indices,
    intervals_to_dataframe,
    map_event_source_indices_to_session_time,
)
from multimodal_sync.modalities.audio import load_audio_source_segment
from multimodal_sync.modalities.video import (
    extract_video_roi_file_trace,
    extract_video_roi_source_trace,
    read_video_file_frame,
)
from multimodal_sync.segments import resolve_data_file
from multimodal_sync.timebase import FrameTimebase, SyncTimebase

## Session Paths And Source File Tables

In [ ]:
session_basepath = Path(os.environ.get("MULTIMODAL_SYNC_EXAMPLE_SESSION", "/path/to/example_session_01"))

# Resolve paths whether the notebook is launched from the repository root or from sync_analysis/notebooks.
if (Path.cwd() / "sync_analysis" / "configs").is_dir():
    sync_analysis_root = Path.cwd() / "sync_analysis"
elif (Path.cwd().parent / "configs").is_dir():
    sync_analysis_root = Path.cwd().parent
else:
    raise FileNotFoundError("Could not locate sync_analysis/configs from the current working directory")
config_path = sync_analysis_root / "configs" / "example_a1v1i_50hz.yaml"

audio_channel_id = "c1_high"
video_channel_id = "cam0"

config = config_root(load_config(config_path))
sync_rate_hz = get_sync_rate_hz(config)
audio_sample_rate_hz = config["audio"]["audio_file_sr"]
video_frame_rate_hz = config["video"]["video_file_sr"]

sync_basepath = session_basepath / "sync"
audio_basepath = session_basepath / config["audio"].get("audio_base_dir", "raw_audio") / audio_channel_id
video_basepath = session_basepath / config["video"].get("video_base_dir", "raw_video") / video_channel_id

audio_file_info = pd.read_csv(sync_basepath / "audio" / audio_channel_id / "audio_file_info.csv", index_col=0)
video_file_info = pd.read_csv(sync_basepath / "video" / video_channel_id / "video_file_info.csv", index_col=0)

display(audio_file_info)
display(video_file_info)

The file-info tables define the source coordinate system for each channel. They specify where each file begins and ends in channel-wide sample or frame indices. Event detection does not need synchronization information; it only needs each source file and its local coordinate system.

## Pick A Video ROI From One Source File

In [ ]:
example_video_row = video_file_info.iloc[0]
example_video_path = resolve_data_file(video_basepath, str(example_video_row["filename"]))
example_file_frame_index = 50
example_frame = read_video_file_frame(
    video_path=example_video_path,
    frame_index=example_file_frame_index,
)

# Update this ROI if using a different video or camera view.
led_roi = {"x": 920, "y": 385, "w": 16, "h": 16}

display_frame = example_frame[..., ::-1] if example_frame.ndim == 3 else example_frame

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(display_frame, cmap="gray")
ax.add_patch(Rectangle((led_roi["x"], led_roi["y"]), led_roi["w"], led_roi["h"], fill=False, color="tab:red"))
ax.set_title(f"{example_video_row['filename']}, local frame {example_file_frame_index}")
ax.set_axis_off()
plt.show()

## Detect Buzzer Events Per Audio File

This step treats the audio channel as ordinary audio data. Each WAV file is loaded and analyzed in file-local time. The detector returns an event table with `start` and `end` times in seconds from the beginning of that WAV file.

In [ ]:
audio_threshold = 0.4
audio_events_by_file_parts = []

for file_index, row in audio_file_info.iterrows():
    audio_path = resolve_data_file(audio_basepath, str(row["filename"]))
    audio_values, observed_sr = sf.read(audio_path, always_2d=False)
    if int(round(observed_sr)) != int(round(audio_sample_rate_hz)):
        raise ValueError(f"Unexpected audio sample rate for {audio_path}: {observed_sr}")
    if audio_values.ndim == 2:
        audio_values = audio_values[:, 0]

    file_local_times_s = np.arange(audio_values.shape[0]) / audio_sample_rate_hz
    audio_amplitude = np.abs(audio_values)

    local_intervals = detect_threshold_intervals(
        file_local_times_s,
        audio_amplitude,
        threshold=audio_threshold,
        direction="above",
        gap_threshold=0.05,
        min_duration=0.05,
        max_duration=2.0,
    )
    file_events = intervals_to_dataframe(local_intervals, start_col="start", end_col="end")
    file_events.insert(0, "filename", row["filename"])
    file_events.insert(1, "file_index", file_index)
    audio_events_by_file_parts.append(file_events)

audio_events_by_file = pd.concat(audio_events_by_file_parts, ignore_index=True)
display(audio_events_by_file)

## Convert Buzzer Events To Audio Source Indices

Now use `audio_file_info.csv` to place the file-local detections into the audio channel's source coordinate system. This creates a per-source event table with `source_start_index` and `source_end_index` sample indices.

In [ ]:
audio_events_source_parts = []

for file_index, row in audio_file_info.iterrows():
    file_events = audio_events_by_file.loc[audio_events_by_file["filename"] == row["filename"]]
    source_events = file_local_time_events_to_source_indices(
        file_events,
        file_source_start_index=int(row["sample_start"]),
        source_rate_hz=audio_sample_rate_hz,
        file_local_start_col="start",
        file_local_end_col="end",
    )
    audio_events_source_parts.append(source_events)

buzzer_events_source = pd.concat(audio_events_source_parts, ignore_index=True)
display(buzzer_events_source)

### Audio Source-Space Plot

The x-axis here is audio source time based on channel-wide audio sample index and the nominal audio sampling rate. No session-time mapping has been applied yet.

In [ ]:
audio_source_start_index = int(audio_file_info["sample_start"].min())
audio_source_end_index = int(audio_file_info["sample_end"].max())

audio_source_full = load_audio_source_segment(
    file_info=audio_file_info,
    audio_basepath=audio_basepath,
    global_start_sample=audio_source_start_index,
    global_end_sample=audio_source_end_index,
    sample_rate_hz=audio_sample_rate_hz,
    sample_channel=0,
    channel_id=audio_channel_id,
)

audio_source_time_s = audio_source_full.source_times_nominal_s
source_event_start_s = buzzer_events_source["source_start_index"] / audio_sample_rate_hz
source_event_end_s = buzzer_events_source["source_end_index"] / audio_sample_rate_hz

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(audio_source_time_s, audio_source_full.values, color="tab:blue")
for start, end in zip(source_event_start_s, source_event_end_s):
    ax.axvspan(start, end, color="tab:blue", alpha=0.15)
ax.set_xlabel("Audio source time from audio channel start (s)")
ax.set_ylabel("Audio amplitude")
plt.tight_layout()
plt.show()

## Detect LED Events Per Video File

This step treats the video channel as ordinary video data. Each video file is analyzed in file-local frame coordinates. The detector returns an event table with `start` and `end` frame indices from the beginning of that video file.

In [ ]:
led_threshold = 98.0
led_events_by_file_parts = []

for file_index, row in video_file_info.iterrows():
    video_path = resolve_data_file(video_basepath, str(row["filename"]))
    led_roi_trace = extract_video_roi_file_trace(video_path=video_path, roi=led_roi)
    file_local_frames = np.arange(led_roi_trace.size)

    local_intervals = detect_threshold_intervals(
        file_local_frames,
        led_roi_trace,
        threshold=led_threshold,
        direction="above",
        gap_threshold=2,
        min_duration=5,
        max_duration=100,
    )
    file_events = intervals_to_dataframe(local_intervals, start_col="start", end_col="end")
    file_events.insert(0, "filename", row["filename"])
    file_events.insert(1, "file_index", file_index)
    led_events_by_file_parts.append(file_events)

led_events_by_file = pd.concat(led_events_by_file_parts, ignore_index=True)
display(led_events_by_file)

## Convert LED Events To Video Source Indices

Now use `video_file_info.csv` to place the file-local detections into the video channel's source coordinate system. This creates a per-source event table with `source_start_index` and `source_end_index` frame indices.

In [ ]:
led_events_source_parts = []

for file_index, row in video_file_info.iterrows():
    file_events = led_events_by_file.loc[led_events_by_file["filename"] == row["filename"]]
    source_events = file_local_index_events_to_source_indices(
        file_events,
        file_source_start_index=int(row["frame_start"]),
        file_local_start_col="start",
        file_local_end_col="end",
    )
    led_events_source_parts.append(source_events)

led_events_source = pd.concat(led_events_source_parts, ignore_index=True)
display(led_events_source)

### Video Source-Space Plot

The x-axis here is video source time based on channel-wide frame index and the nominal video frame rate. No session-time mapping has been applied yet.

In [ ]:
video_source_start_index = int(video_file_info["frame_start"].min())
video_source_end_index = int(video_file_info["frame_end"].max())

led_source_full = extract_video_roi_source_trace(
    file_info=video_file_info,
    video_basepath=video_basepath,
    global_start_frame=video_source_start_index,
    global_end_frame=video_source_end_index,
    frame_rate_hz=video_frame_rate_hz,
    roi=led_roi,
    channel_id=video_channel_id,
)

video_source_time_s = led_source_full.source_times_nominal_s
video_event_start_s = led_events_source["source_start_index"] / video_frame_rate_hz
video_event_end_s = led_events_source["source_end_index"] / video_frame_rate_hz

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(video_source_time_s, led_source_full.values, color="tab:orange")
for start, end in zip(video_event_start_s, video_event_end_s):
    ax.axvspan(start, end, color="tab:orange", alpha=0.2)
ax.set_xlabel("Video source time from video channel start (s)")
ax.set_ylabel("LED ROI mean intensity")
plt.tight_layout()
plt.show()

## Build Source-To-Session Timebases

Only now do we use synchronization information. The audio timebase comes from the sync pulses embedded in the audio source channel. The video timebase comes from the hardware-triggered design: one video frame per sync pulse.

In [ ]:
audio_sync_data = np.load(sync_basepath / "audio" / audio_channel_id / "audio_sync_data.npy")
audio_timebase = SyncTimebase.from_sync_data(
    audio_sync_data,
    source_rate_hz=audio_sample_rate_hz,
    source_index_name="sample",
)
video_timebase = FrameTimebase(sync_rate_hz=sync_rate_hz)

session_duration_s = (audio_sync_data[-1, 3] + 1) / sync_rate_hz
print(f"Session duration: {session_duration_s:.3f} s")
print(audio_timebase.to_dict())

## Map Source Signals And Events To Session Time

The source traces and source event tables are now mapped once into the shared session timebase. Because source files can include pre-session and post-session data, the mapped event table may contain events outside the sync-defined session. For session-level comparisons, keep the events that overlap the session interval.

In [ ]:
audio_session_signal = map_source_signal_to_session_time(audio_source_full, audio_timebase)
led_session_signal = map_source_signal_to_session_time(led_source_full, video_timebase)

buzzer_events_session = map_event_source_indices_to_session_time(buzzer_events_source, audio_timebase)
led_events_session = map_event_source_indices_to_session_time(led_events_source, video_timebase)

buzzer_events_session_in_session = buzzer_events_session.loc[
    (buzzer_events_session["session_end_s"] >= 0) &
    (buzzer_events_session["session_start_s"] <= session_duration_s)
].copy()
led_events_session_in_session = led_events_session.loc[
    (led_events_session["session_end_s"] >= 0) &
    (led_events_session["session_start_s"] <= session_duration_s)
].copy()

display(buzzer_events_session)
display(led_events_session)
print(f"Session-overlapping events: buzzer={len(buzzer_events_session_in_session)}, LED={len(led_events_session_in_session)}")

## Compare Buzzer And LED Events On The Session Clock

In [ ]:
n_compare = min(len(buzzer_events_session_in_session), len(led_events_session_in_session))
event_comparison = pd.DataFrame({
    "buzzer_session_start_s": buzzer_events_session_in_session["session_start_s"].to_numpy()[:n_compare],
    "led_session_start_s": led_events_session_in_session["session_start_s"].to_numpy()[:n_compare],
    "buzzer_session_end_s": buzzer_events_session_in_session["session_end_s"].to_numpy()[:n_compare],
    "led_session_end_s": led_events_session_in_session["session_end_s"].to_numpy()[:n_compare],
})
event_comparison["start_delta_s"] = event_comparison["buzzer_session_start_s"] - event_comparison["led_session_start_s"]
event_comparison["end_delta_s"] = event_comparison["buzzer_session_end_s"] - event_comparison["led_session_end_s"]
event_comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

audio_plot = audio_session_signal.values / np.max(np.abs(audio_session_signal.values))
led_plot = (led_session_signal.values - np.min(led_session_signal.values)) / (
    np.max(led_session_signal.values) - np.min(led_session_signal.values)
)

ax.plot(audio_session_signal.session_times_s, audio_plot, color="tab:blue", label="audio waveform")
ax.plot(led_session_signal.session_times_s, led_plot + 1.5, color="tab:orange", label="LED ROI intensity")

for _, row in buzzer_events_session_in_session.iterrows():
    ax.axvspan(row["session_start_s"], row["session_end_s"], color="tab:blue", alpha=0.15)
for _, row in led_events_session_in_session.iterrows():
    ax.axvspan(row["session_start_s"], row["session_end_s"], color="tab:orange", alpha=0.15)

ax.set_xlim(0, session_duration_s)
ax.set_xlabel("Session time (s)")
ax.set_ylabel("Normalized signal")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()